# Module 3.3: Random Forests & Gradient Boosting (XGBoost / LightGBM)
### 🇵🇱 Lasy Losowe i Gradient Boosting w Danych Tabularnych z Czujników

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/max480425/industrial-ml-course/blob/main/public/notebooks/m03_l03_random_forest_xgboost_lightgbm.ipynb)

**Industrial Machine Learning & Predictive Maintenance Interactive Laboratory**

> **Summary:** Deploy Random Forests, XGBoost, and LightGBM on multi-sensor telemetry to benchmark defect classification accuracy, training speed, and feature importance.  
> 🇵🇱 **Streszczenie:** Wdróż Lasy Losowe, XGBoost i LightGBM na danych z czujników, porównując precyzję detekcji wad, czas trenowania i wagę cech.

- **Estimated Time:** 30 minutes
- **Runtime:** Python 3.10+ (CPU or Free Tier T4 GPU)
- **Packages:** `numpy`, `scipy`, `scikit-learn`, `pandas`, `matplotlib`, `seaborn`
---


In [ ]:
# Step 1: Install & Verify Python Scientific Environment
!pip install -q numpy scipy scikit-learn pandas matplotlib seaborn

import numpy as np
import pandas as pd
import scipy.signal as signal
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'monospace'
print("✓ Industrial ML environment loaded successfully.")


### Key Engineering Concepts:
1. **Ensemble Methods for Sensor Arrays**: Factory telemetry features contain non-linear interactions and sensor dropouts. Random Forests and Gradient Boosted Trees are the industrial gold standard.
2. **Handling Severe Imbalance**: Machine breakdowns are rare events ($1\%-3\%$ of samples). Using `scale_pos_weight` in XGBoost/LightGBM forces the loss function to heavily penalize missed failures.
3. **Feature Importance**: Comparing Gini Impurity reduction against Permutation Importance to identify true root causes without bias toward high-cardinality sensors.


In [ ]:
# 1. Synthesize 25-Sensor Packaging Line Failure Dataset
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

np.random.seed(42)
n_cycles = 2500
n_sensors = 25

# Normal operational telemetry
X_factory = np.random.normal(loc=50.0, scale=8.0, size=(n_cycles, n_sensors))

# True failure condition triggered by interaction of Sensor 3 and Sensor 17
y_failure = np.zeros(n_cycles, dtype=int)
risk_score = 0.6 * (X_factory[:, 3] - 50.0) + 0.8 * (X_factory[:, 17] - 50.0)
failure_prob = 1.0 / (1.0 + np.exp(-0.25 * risk_score))
y_failure[failure_prob > 0.85] = 1

print(f"✓ Synthetic line output: {n_cycles} cycles, {y_failure.sum()} breakdown events ({(y_failure.mean()*100):.1f}% failure rate).")


In [ ]:
# 2. Fit Random Forest & Evaluate Permutation Feature Importance
from sklearn.inspection import permutation_importance

rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf.fit(X_factory, y_failure)

perm_imp = permutation_importance(rf, X_factory, y_failure, n_repeats=5, random_state=42)
top_indices = np.argsort(perm_imp.importances_mean)[::-1][:6]

plt.figure(figsize=(10, 4.5))
plt.barh([f"Sensor {i}" for i in top_indices][::-1], perm_imp.importances_mean[top_indices][::-1], color='#10b981')
plt.title("Permutation Feature Importance (Identifies Root Cause Sensors)", fontweight='bold')
plt.xlabel("Mean Accuracy Degradation Upon Feature Shuffling")
plt.grid(axis='x', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

auc = roc_auc_score(y_failure, rf.predict_proba(X_factory)[:, 1])
print(f"Random Forest ROC-AUC Score: {auc:.4f}")


## 📋 Practical Hands-On Checklist

- [ ] **Task 1:** Train a Random Forest classifier with balanced class weights on imbalanced defect logs *(🇵🇱 Wytrenuj klasyfikator Lasu Losowego z wagami balanced na niezbalansowanym zbiorze awarii)*
- [ ] **Task 2:** Fit XGBoost and LightGBM models with histogram-based binning *(🇵🇱 Wytrenuj modele XGBoost i LightGBM z dyskretyzacją histogramową cech)*
- [ ] **Task 3:** Extract and plot Gini and SHAP feature importance rankings to identify leading failure indicators *(🇵🇱 Wyznacz i wykreśl ranking istotności cech (SHAP i Gini), identyfikując główne wskaźniki awarii)*


In [ ]:
# Exercise Challenge:
# Verify that Sensor 3 and Sensor 17 are ranked as the top 2 most important features
assert 3 in top_indices[:2] and 17 in top_indices[:2], "Permutation importance failed to isolate the root cause sensors."
print("✓ Ensemble feature importance verified.")
